In [1]:
!pip install -q spacy networkx
!python -m spacy download en_core_web_sm -q
import spacy
import networkx as nx
import pandas as pd
from IPython.display import display
from spacy import displacy
nlp = spacy.load("en_core_web_sm")
sentence = "Google acquired a startup in Pune."
doc = nlp(sentence)
print("SENTENCE:", sentence)

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
SENTENCE: Google acquired a startup in Pune.


In [2]:
# ---------------- 1. NAMED ENTITY RECOGNITION ---------------
print("\n" + "=" * 60)
print("1) NAMED ENTITIES")
print("=" * 60)
ent_rows = [{"Entity": e.text, "Type": e.label_,
"Meaning": spacy.explain(e.label_)} for e in doc.ents]
display(pd.DataFrame(ent_rows))
# "startup" is a common noun, so NER skips it, but it is an important
# participant in the relation, so we also list the noun chunks.
print("Noun chunks (candidate participants):", [c.text for c in doc.noun_chunks])


1) NAMED ENTITIES


,Entity,Type,Meaning
0,Google,ORG,"Companies, agencies, institutions, etc."
1,Pune,LOC,"Non-GPE locations, mountain ranges, bodies of ..."


Noun chunks (candidate participants): ['Google', 'a startup', 'Pune']


In [3]:
# ---------------- 2. DEPENDENCY PARSE ---------------
print("\n" + "=" * 60)
print("2) DEPENDENCY PARSE")
print("=" * 60)
dep_rows = [{"Token": t.text, "POS": t.pos_, "Dependency": t.dep_,
"Head": t.head.text, "Children": [c.text for c in t.children]}
for t in doc]
display(pd.DataFrame(dep_rows))
displacy.render(doc, style="dep", jupyter=True, options={"distance": 110})



2) DEPENDENCY PARSE


,Token,POS,Dependency,Head,Children
0,Google,PROPN,nsubj,acquired,[]
1,acquired,VERB,ROOT,acquired,"[Google, startup, in, .]"
2,a,DET,det,startup,[]
3,startup,NOUN,dobj,acquired,[a]
4,in,ADP,prep,acquired,[Pune]
5,Pune,PROPN,pobj,in,[]
6,.,PUNCT,punct,acquired,[]


In [4]:
# ---------------- 3. RELATION EXTRACTION ---------------
print("\n" + "=" * 60)
print("3) EXTRACTED RELATIONS (subject, relation, object)")
print("=" * 60)
triples = []
# Rule A: subject - verb - direct object
for tok in doc:
    if tok.pos_ == "VERB":
        subj = [c for c in tok.children if c.dep_ in ("nsubj", "nsubjpass")]
        obj = [c for c in tok.children if c.dep_ in ("dobj", "obj")]
        for s in subj:
            for o in obj:
                triples.append((s.text, tok.lemma_.upper(), o.text))
# Rule B: noun/verb - preposition - place entity (GPE/LOC)
for tok in doc:
    if tok.dep_ == "prep":
        for c in tok.children:
            if c.dep_ == "pobj" and c.ent_type_ in ("GPE", "LOC"):
                head = tok.head
                if head.pos_ == "VERB":
                    triples.append((head.text + " (event)", "LOCATED_IN", c.text))
                else:
                    triples.append((head.text, "LOCATED_IN", c.text))
rel_df = pd.DataFrame(triples, columns=["Subject", "Relation", "Object"])
display(rel_df)
for s, r, o in triples:
    print(f"  ({s})  --[{r}]-->  ({o})")


3) EXTRACTED RELATIONS (subject, relation, object)


,Subject,Relation,Object
0,Google,ACQUIRE,startup
1,acquired (event),LOCATED_IN,Pune


  (Google)  --[ACQUIRE]-->  (startup)
  (acquired (event))  --[LOCATED_IN]-->  (Pune)


In [6]:
# ---------------- 4. DEPENDENCY PATH BETWEEN ENTITIES ---------------
print("\n" + "=" * 60)
print("4) DEPENDENCY PATH BETWEEN THE ENTITIES")
print("=" * 60)

# Build an (undirected) graph of the parse tree so we can find the shortest path
graph = nx.Graph()
for tok in doc:
    for child in tok.children:
        graph.add_edge(tok.i, child.i)

def dependency_path(doc, i, j):
    """Return the path between token i and token j with arrow directions."""
    nodes = nx.shortest_path(graph, i, j)
    parts = [doc[nodes[0]].text]
    for a, b in zip(nodes, nodes[1:]):
        if doc[b].head.i == a:
            # moving down: a is head of b
            parts.append(f"-{doc[b].dep_}->")
        else:
            # moving up: b is head of a
            parts.append(f"<-{doc[a].dep_}-")
        parts.append(doc[b].text)
    return " ".join(parts), [doc[n].text for n in nodes]

ents = list(doc.ents)
for x in range(len(ents)):
    for y in range(x + 1, len(ents)):
        e1, e2 = ents[x], ents[y]
        path_str, path_words = dependency_path(doc, e1.root.i, e2.root.i)
        print(f"\n{e1.text} ({e1.label_})  ->  {e2.text} ({e2.label_})")
        print("  Path        :", path_str)
        print("  Path length :", len(path_words) - 1, "edges")

# Path to the noun "startup" as well (it links the two entities)
startup = [t for t in doc if t.text.lower() == "startup"][0]
google = ents[0].root
path_str, _ = dependency_path(doc, google.i, startup.i)
print("\nGoogle -> startup")
print("  Path        :", path_str)


4) DEPENDENCY PATH BETWEEN THE ENTITIES

Google (ORG)  ->  Pune (LOC)
  Path        : Google <-nsubj- acquired -prep-> in -pobj-> Pune
  Path length : 3 edges

Google -> startup
  Path        : Google <-nsubj- acquired -dobj-> startup


In [8]:
# ---------------- 5. HOW THE PATH BECOMES A RELATION ---------------
print("\n" + "=" * 60)
print("5) PATH PATTERN -> RELATION")
print("=" * 60)
print("Pattern  : ORG <-nsubj- VERB -dobj-> NOUN  ==>  ORG ACQUIRES NOUN")
print("Pattern  : NOUN -prep-> in -pobj-> GPE     ==>  NOUN LOCATED_IN GPE")
print("Combined : Google ACQUIRED a startup that is LOCATED_IN Pune")


5) PATH PATTERN -> RELATION
Pattern  : ORG <-nsubj- VERB -dobj-> NOUN  ==>  ORG ACQUIRES NOUN
Pattern  : NOUN -prep-> in -pobj-> GPE     ==>  NOUN LOCATED_IN GPE
Combined : Google ACQUIRED a startup that is LOCATED_IN Pune
